# Event pairing benchmark

**Question:** when two sources describe the same disaster, which method notices?

`monty:corr_id` cannot join sources (two sources give one disaster different ids), so a matcher has to.
This notebook scores three candidates on a small hand-labelled gold set of real Montandon events:

| Scorer | What it is |
|---|---|
| **Notebook-factory heuristic** | the experimental time-and-distance score in `factory/events/disasters.py` (not an authoritative Montandon method), with its shared-ID shortcuts removed so it is judged on evidence alone |
| **Learned** | logistic regression on the same evidence plus hazard relation and place-name overlap, scored out-of-sample (each window by a model trained on the others) |
| **Rules** | the other matching methods used across Montandon, as yes/no rows: `corr_id` equality (skills), date + country from the `corr_id` (MCP toolset `event_key`) and the extension docs' CQL2 "Algorithm 1" (shared hazard code, shared country, dates within 1 or 7 days) |
| **Listwise decision model** | the same model, asked once per record and source about all of that source's candidates together, so it can compare near neighbours (`listwise` parameter) |
| **Decision model** | a System One model (e.g. Jev) via OpenRouter's Decisions API, asked from the records' words, dates and distances whether they are the same disaster. Needs `OPENROUTER_API_KEY` |

A fourth row, **heuristic → model**, is the proposed architecture: the heuristic settles clear pairs and the
model is asked only about the band between `band_low` and `band_high`. Widen the band if the heuristic is
confidently wrong outside it (the notebook prints how many pairs it settles wrongly).

**How to read it.** Labels come from `gold_disasters.yml`: grouped by an AI agent from the records and public
knowledge, never from a shared ID, and not yet reviewed by a person. They agree with every USGS id and GLIDE
number GDACS cites for its own events. A second set, `silver_disasters.yml`, is labelled by those GDACS
references alone. The headline counts only pairs with a record from an impact source (a register, a response
operation, displacement or damage mapping): destructive events, not every small quake. The gold set is small
(about 200 records in 36 disasters), so read the 95% intervals and look for *which* pairs each scorer gets wrong. Thresholds marked *in-sample* are tuned on
the data they are scored on and are optimistic. Pairs are not independent (a record sits in many pairs), so
read the numbers as indicative.

Extend the gold set by adding item ids to the YAML; the notebook re-scores on every run. The run's catalog
comes from the Montandon event you select; the event itself is not used.

In [ ]:
# --- Reserved parameters (injected by the Notebook Factory; defaults keep the notebook runnable standalone) ---
run_id = ""
inputs_dir = ""
output_dir = "outputs"
area = {}                      # {id, code, name, level, country, country_name, bbox}
area_geojson = ""              # FeatureCollection with the target area
subdivisions_geojson = ""      # FeatureCollection with the aggregation subdivisions
stac_item = {}                 # the selected STAC item (dict), only its catalog is used
stac_item_json = ""
stac_collection = ""
stac_api_url = ""              # root of the item's catalog (Montandon)
stac_catalog_profile = ""      # "generic" | "montandon"
filters = {}                   # the run's filter tags
hazard = ""

# --- User parameters ---
block_days = 45                # candidates are records at most this many days apart
decision_models = "typesafe/jev-1.14-20261014"   # comma-separated OpenRouter model ids, empty to skip
max_model_calls = 3000         # cap on Decisions API calls (one per candidate pair per model)
l2 = 1.0                       # ridge strength of the learned scorer
band_low = 0.4                 # hybrid: heuristic scores below this are settled as different
band_high = 0.7                # hybrid: heuristic scores at or above this are settled as same
gate_days = 7                  # pipeline: pairs more than this many days apart are settled as different
margin = 0.15                  # pipeline: keep a pair only if within this of its records' best candidate per source
listwise = True                # also score with one call per record and source, all its candidates together
max_candidates = 12            # most candidates in one listwise call
distractors = 0                # unscored catalog records added to each listwise call as realistic rivals
describe_detail = True         # send the model each record's magnitude and end date
bootstrap_samples = 1000       # resamples of whole windows for the 95% intervals
silver = True                  # also score silver_disasters.yml (labels from GDACS's own references)
stability_pairs = 100          # pairs asked again, and with the records swapped, to measure stability; 0 skips
scale_sample_days = 7          # cost at catalog scale: days of catalog sampled; 0 skips
scale_sample_end = "2025-09-15"  # last day of that sample


## 1. Gold set: the disasters and their records

In [ ]:
import json
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

sys.path.insert(0, str(Path.cwd()))  # the helper module ships next to the notebook
import pairing as P

P.DESCRIBE_DETAIL = describe_detail

warnings.filterwarnings("ignore")
Path(output_dir).mkdir(parents=True, exist_ok=True)

gold = yaml.safe_load(Path("gold_disasters.yml").read_text())
label, windows, kinds = P.gold_labels(gold)

api_url = stac_api_url or os.environ.get("STAC_API_URL", "")
if not api_url:
    raise RuntimeError("No catalog: select a Montandon event when starting the run (it supplies the catalog URL).")
items = P.fetch_items(api_url, os.environ.get("STAC_API_KEY", ""), list(label))
missing = sorted(set(label) - set(items))
label = {i: d for i, d in label.items() if i in items}
records = {i: P.record_from_item(x) for i, x in items.items()}

coverage = (
    pd.DataFrame({"disaster": [label[i] for i in records], "window": [windows[label[i]] for i in records]})
    .groupby(["window", "disaster"]).size().rename("records").reset_index()
)
print(f"Catalog: {api_url}")
print(f"{len(items)} of {len(items) + len(missing)} gold records found in the catalog; "
      f"{len(gold['ambiguous'])} more are listed as ambiguous and never scored.")
if missing:
    print("MISSING from the catalog (re-ingested or removed? update gold_disasters.yml):", ", ".join(missing))
display(coverage)


## 2. Candidate pairs

Stage 1 of any matcher is cheap and deterministic: only records that share a country (an unknown country never
blocks) and are at most `block_days` apart are candidates. Pairs inside one source family are skipped, and IDMC's
two collections count as one family. Positives are pairs from the same gold disaster; negatives are pairs from
different disasters of the same window (same country and period), which is what makes them hard.

In [ ]:
pairs = pd.DataFrame(P.build_pairs(records, gold, block_days))


def known(r):
    return set(r.countries) - {P.UNKNOWN_COUNTRY}


pairs["cross_border"] = [bool(known(records[a]) and known(records[b]) and not known(records[a]) & known(records[b]))
                         for a, b in zip(pairs.a, pairs.b, strict=True)]
pairs["in_scope"] = [P.destructive(records[a]) or P.destructive(records[b]) for a, b in zip(pairs.a, pairs.b, strict=True)]
copies = P.copy_counts(records, label)
pairs["weight"] = P.pair_weights(pairs.a, pairs.b, copies)
total_pos = int(pairs.y.sum())
lost = pairs[(pairs.y == 1) & (pairs.block != "ok")]
blocked = pairs[pairs.block == "ok"].reset_index(drop=True)
y = blocked.y.to_numpy()
w = blocked.weight.to_numpy()
pos_weight = float(pairs.weight[pairs.y == 1].sum())

display(pd.DataFrame({
    "pairs": [len(pairs), len(blocked), int(blocked.in_scope.sum())],
    "positive": [total_pos, int(y.sum()), int(y[blocked.in_scope].sum())],
    "negative": [int((pairs.y == 0).sum()), int((y == 0).sum()), int((y[blocked.in_scope] == 0).sum())],
}, index=["all pairs in a window", "candidates (pass blocking)", "candidates in scope (an impact source)"]))
print(f"Blocking recall: {1 - len(lost) / max(total_pos, 1):.1%} "
      f"({len(lost)} positives lost; by reason: {lost.block.value_counts().to_dict() or 'none'})")
print(f"Identical copies (IDMC IDU/GIDD, DesInventar rows) are weighted so each pair counts once: "
      f"{int((w < 1).sum())} candidate pairs carry a weight below 1.")
display(blocked.groupby("pair_type").y.agg(pairs="size", positives="sum"))
display(blocked.groupby("hazard_rel").y.agg(pairs="size", positives="sum")
        .rename_axis("hazard codes of the two records"))

## 3. Scorers

The learned scorer's weights (standardised features, fit on all candidates) show which evidence carries the
decision; positive means "more likely the same disaster".

In [ ]:
NF_HEURISTIC, LEARNED, HYBRID = "Notebook-factory heuristic", "Learned", "Heuristic → model"
scores = {NF_HEURISTIC: blocked.heuristic.astype(float).to_numpy()}
default_threshold = {NF_HEURISTIC: P.STRONG, LEARNED: 0.5}

X = blocked[[f"f_{k}" for k in P.FEATURES]].to_numpy(float)
scores[LEARNED] = P.cross_val_logreg(X, y, blocked.window.to_numpy(), l2)
for rule_name, rule in P.RULES.items():
    scores[rule_name] = np.array([rule(records[a], records[b]) for a, b in zip(blocked.a, blocked.b, strict=True)], dtype=float)
    default_threshold[rule_name] = 0.5

weights = P.fit_logreg(X, y, l2).weights[1:]
display(pd.Series(weights, index=P.FEATURES, name="weight").sort_values(key=abs, ascending=False).round(2).to_frame())


In [ ]:
import time

api_key = os.environ.get("OPENROUTER_API_KEY", "")
models = [m.strip() for m in decision_models.split(",") if m.strip()]
model_names, model_of, model_log = [], {}, []

if not models:
    print("No decision model requested: skipping.")
elif not api_key:
    print("OPENROUTER_API_KEY is not set: skipping the decision model.")
elif len(blocked) * len(models) > max_model_calls:
    print(f"{len(blocked) * len(models)} calls needed, above max_model_calls={max_model_calls}: skipping.")
else:
    pair_records = [(records[a], records[b]) for a, b in zip(blocked.a, blocked.b, strict=True)]
    for model in models:
        started = time.time()
        results = P.score_with_model(pair_records, model, api_key)
        name = f"Model {model.split('/')[-1]}"
        model_names.append(name)
        model_of[name] = model
        blocked[f"p_same|{model}"] = [r["p_same"] for r in results]
        blocked[f"relation|{model}"] = [r.get("relation") for r in results]
        scores[name] = blocked[f"p_same|{model}"].astype(float).to_numpy()
        default_threshold[name] = 0.5
        errors = [r["error"] for r in results if r.get("error")]
        served = next((r["model"] for r in results if r.get("model")), None)
        model_log.append({"requested": model, "served": served, "pairs": len(results), "errors": len(errors),
                          "cost": round(sum(r.get("cost") or 0 for r in results), 5),
                          "seconds": round(time.time() - started)})
        if errors:
            print(f"{model}: {len(errors)} calls failed, e.g. {errors[0]}")
    display(pd.DataFrame(model_log))


In [ ]:
# The proposed architecture: the heuristic settles clear pairs, the model is asked about the band between.
h = scores[NF_HEURISTIC]
for name in model_names:
    p = scores[name]
    hybrid = np.where(h >= band_high, 1.0, np.where(h < band_low, 0.0, np.where(np.isnan(p), h, p)))
    scores[f"{HYBRID} {name.removeprefix('Model ')}"] = hybrid
    default_threshold[f"{HYBRID} {name.removeprefix('Model ')}"] = 0.5

band = (h >= band_low) & (h < band_high)
settled_wrong = int((((h >= band_high) & (y == 0)) | ((h < band_low) & (y == 1))).sum())
print(f"Escalated to the model: heuristic between {band_low} and {band_high}, {int(band.sum())} of {len(h)} candidates "
      f"({int(y[band].sum())} positive). Settled by the heuristic but wrong: {settled_wrong}, which no model can fix.")


## Listwise decision model

One request per record and per other source: the record (the anchor) and every candidate from that source that
passed the date gate, with a yes/no per candidate and the single best one. Each pair is asked from both sides and
the two answers averaged. Pairs beyond `gate_days` are never asked (settled as different). With `distractors`
above 0, unlabelled records from the catalog (same source, same countries, within the gate) join each request as
realistic rivals; they are never scored.

In [ ]:
from datetime import timedelta

gap = blocked.gap_days.to_numpy(float)
listwise_names, listwise_log = [], []
if listwise and model_names:
    in_gate = gap <= gate_days
    gated_pairs = list(zip(blocked.a[in_gate], blocked.b[in_gate], strict=True))
    extra, lookup = {}, dict(records)
    if distractors:
        context = {}
        for window_id in sorted(set(windows.values())):
            members = [r for i, r in records.items() if windows[label[i]] == window_id]
            countries = {c for r in members for c in r.countries} - {P.UNKNOWN_COUNTRY}
            starts = [r.start for r in members if r.start]
            if not countries or not starts:
                continue
            slack = timedelta(days=gate_days)
            ends = [r.end or r.start for r in members if r.start]
            for item in P.fetch_context(api_url, os.environ.get("STAC_API_KEY", ""), countries, min(starts) - slack, max(ends) + slack):
                if item["id"] not in records and item["id"] not in context:
                    context[item["id"]] = P.record_from_item(item)
        keys = sorted({(a, records[b].family) for a, b in gated_pairs} | {(b, records[a].family) for a, b in gated_pairs})
        extra = P.choose_distractors(keys, records, context, gate_days, distractors)
        lookup = {**context, **records}
        print(f"{len(context)} unlabelled catalog records fetched; {sum(len(v) for v in extra.values())} added to requests as rivals.")
    groups = P.listwise_groups(gated_pairs, records, max_candidates, extra)
    if len(groups) * len(model_names) > max_model_calls:
        print(f"{len(groups) * len(model_names)} listwise calls needed, above max_model_calls={max_model_calls}: skipping.")
    else:
        pair_keys = list(zip(blocked.a, blocked.b, strict=True))
        for name in model_names:
            started = time.time()
            results = P.score_listwise(groups, lookup, model_of[name], api_key)
            score, best = P.pair_scores_from_listwise(groups, results, pair_keys)
            lw = f"Listwise {name.removeprefix('Model ')}"
            scores[lw] = np.where(gap > gate_days, 0.0, score)
            default_threshold[lw] = 0.5
            listwise_names.append(lw)
            blocked[f"listwise_best|{model_of[name]}"] = np.where(gap > gate_days, np.nan, best)
            errors = [r["error"] for r in results if r.get("error")]
            listwise_log.append({"requested": model_of[name], "requests": len(groups),
                                 "candidates per request": round(float(np.mean([len(g[1]) for g in groups])), 1),
                                 "errors": len(errors), "cost": round(sum(r.get("cost") or 0 for r in results), 5),
                                 "seconds": round(time.time() - started)})
            if errors:
                print(f"{name}: {len(errors)} listwise calls failed, e.g. {errors[0]}")
        display(pd.DataFrame(listwise_log))


## 4. Results

*Average precision* ignores thresholds: 1.0 means every same-disaster pair outranks every different-disaster
pair. *Precision / recall / F1* are at each scorer's default threshold (heuristic 0.7, as in notebook-factory; the others
0.5). *End-to-end recall* counts positives lost to blocking as misses.

In [ ]:
def evaluate(name, s=None, thr=None):
    s = scores[name] if s is None else s
    thr = default_threshold[name] if thr is None else thr
    ok = ~np.isnan(s)
    p, r, f = P.prf(y[ok], s[ok] >= thr, w[ok])
    best_thr, best = P.best_f1(y[ok], s[ok], w[ok])
    linked = (s >= thr) & ok
    return {"scorer": name, "pairs scored": int(ok.sum()), "average precision": P.average_precision(y[ok], s[ok], w[ok]),
            "threshold": thr, "precision": p, "recall": r, "F1": f,
            "end-to-end recall": float(w[linked & (y == 1)].sum()) / max(pos_weight, 1e-9),
            "false merges": int((linked & (y == 0)).sum()),
            "best F1 (in-sample)": best, "at threshold": best_thr}

metrics = pd.DataFrame([evaluate(n) for n in scores]).set_index("scorer")
display(metrics.round(3))

### Pipeline stages: date gate and candidate ranking

`+ gate` settles pairs more than `gate_days` apart as different before the scorer sees them. `+ margin` then
ranks candidates against each other: a pair keeps its score only when it is within `margin` of the best score
either record has in the other's source (several matches per source can survive, so it is not one-to-one). The
listwise rows are gated by construction.

In [ ]:
variants, variant_threshold = {}, {}
a_, b_, fa_, fb_ = (blocked[c].to_numpy() for c in ("a", "b", "source_a", "source_b"))
order = []
for base in [NF_HEURISTIC, *model_names, *listwise_names]:
    s, thr = scores[base], default_threshold[base]
    order.append(base)
    if base not in listwise_names:
        s = P.gate_scores(s, gap, gate_days)
        variants[f"{base} + gate"], variant_threshold[f"{base} + gate"] = s, thr
        order.append(f"{base} + gate")
    ranked_name = f"{base} + gate + margin" if base not in listwise_names else f"{base} + margin"
    variants[ranked_name] = P.margin_scores(s, a_, b_, fa_, fb_, margin)
    variant_threshold[ranked_name] = thr
    order.append(ranked_name)

settled = gap > gate_days
print(f"Date gate ({gate_days} days) settles {int(settled.sum())} of {len(gap)} candidates as different; "
      f"true pairs lost to it: {int((settled & (y == 1)).sum())}. Largest gap between two records of one disaster: "
      f"{gap[y == 1].max():.0f} days.")
pipeline = pd.DataFrame([evaluate(n, *( (scores[n], default_threshold[n]) if n in scores else (variants[n], variant_threshold[n]) ))
                         for n in order]).set_index("scorer")
display(pipeline[["pairs scored", "average precision", "precision", "recall", "F1", "false merges"]].round(3))


### Held-out windows

Windows flagged `heldout` in `gold_disasters.yml` were labelled before any scorer was run on them. Judge a scorer
there first: the original windows were read, and in places relabelled, after earlier results. *Recall* is
end-to-end (positives lost to blocking count as misses).

In [ ]:
held_windows = {d["window"] for d in gold["disasters"] if d.get("heldout")}
is_held = blocked.window.isin(held_windows).to_numpy()
pos_total = {flag: int(((pairs.y == 1) & (pairs.window.isin(held_windows) == flag)).sum()) for flag in (False, True)}
rows = []
all_scores, all_thr = {**scores, **variants}, {**default_threshold, **variant_threshold}
for name in order:
    s, thr, row = all_scores[name], all_thr[name], {"scorer": name}
    for group, flag in (("original", False), ("held-out", True)):
        m = (is_held == flag) & ~np.isnan(s)
        row[f"F1 {group}"] = P.prf(y[m], s[m] >= thr, w[m])[2]
        row[f"recall {group}"] = ((s >= thr) & m & (y == 1)).sum() / max(pos_total[flag], 1)
        row[f"false merges {group}"] = int(((s >= thr) & m & (y == 0)).sum())
    rows.append(row)
display(pd.DataFrame(rows).set_index("scorer").round(3))
print(f"original windows: {int((~is_held).sum())} candidate pairs ({int(y[~is_held].sum())} positive); "
      f"held-out windows: {int(is_held.sum())} ({int(y[is_held].sum())} positive); "
      f"positives lost to blocking: {int(lost.window.isin(held_windows).sum())} held-out, {int((~lost.window.isin(held_windows)).sum())} original")


## The comparison: in scope, with 95% intervals

The pairs that matter for the catalog: at least one record comes from a source that only records a disaster
when it hurt people or property (EM-DAT, GLIDE, IFRC, IDMC, DesInventar, Copernicus EMS, the Charter). A small
offshore quake that only USGS and GDACS report needs no pairing. Identical copies count once.

Intervals come from resampling whole windows `bootstrap_samples` times; on a gold set this small they are wide.
*Δ F1* is each scorer minus the best rule in use today, on the same resamples, so its interval says whether the
gain is more than noise. Read the held-out rows first: those windows were labelled before any scorer saw them
(though the review of 2026-11-12 changed some of their labels after scoring; see the gold file's log).

In [ ]:
scope = blocked.in_scope.to_numpy()
compared = [n for n in [*P.RULES, NF_HEURISTIC, LEARNED, *model_names, *listwise_names] if n in scores]
windows_of = blocked.window.to_numpy()
best_rule = max(P.RULES, key=lambda n: P.prf(y[scope], scores[n][scope] >= 0.5, w[scope])[2])
headline_rows = []
for group, mask in (("held-out windows", scope & is_held), ("all windows", scope)):
    idx = np.flatnonzero(mask)
    samples = P.bootstrap_f1(y[idx], {n: (scores[n][idx], default_threshold[n]) for n in compared},
                             windows_of[idx], w[idx], n=bootstrap_samples)
    f_rule = P.prf(y[idx], scores[best_rule][idx] >= 0.5, w[idx])[2]
    for n in compared:
        s, thr = scores[n][idx], default_threshold[n]
        ok = ~np.isnan(s)
        p_, r_, f_ = P.prf(y[idx][ok], s[ok] >= thr, w[idx][ok])
        lo, hi = P.interval(samples[n])
        dlo, dhi = P.interval(samples[n] - samples[best_rule])
        headline_rows.append({
            "windows": group, "scorer": n, "pairs": int(ok.sum()), "positives": int(y[idx][ok].sum()),
            "precision": round(p_, 3), "recall": round(r_, 3), "F1": round(f_, 3), "F1 95% interval": f"{lo:.2f} to {hi:.2f}",
            "false merges": int(((s >= thr) & ok & (y[idx] == 0)).sum()),
            "Δ F1 vs best rule": round(f_ - f_rule, 3), "Δ F1 95% interval": f"{dlo:+.2f} to {dhi:+.2f}",
        })
headline = pd.DataFrame(headline_rows)
print(f"Best rule in use today (all in-scope windows): {best_rule}")
for group, table in headline.groupby("windows", sort=False):
    print(group)
    display(table.drop(columns="windows").set_index("scorer"))

### Across borders

One disaster has a record per country in EM-DAT and GLIDE, and cyclones cross borders. These are the positive
pairs whose records name no common country. Blocking lets them through when the records are within the
hazard's radius (1,500 km at least for records located only at country level); the rest are lost before any
scorer sees them.

In [ ]:
xb_all = pairs[(pairs.y == 1) & pairs.cross_border]
xb = blocked.cross_border.to_numpy() & (y == 1)
print(f"{len(xb_all)} positive pairs across borders; {int((xb_all.block != 'ok').sum())} lost to blocking "
      f"(country-level records whose country centroids are too far apart).")
display(pd.DataFrame({n: {"linked": int(((scores[n] >= default_threshold[n]) & ~np.isnan(scores[n]) & xb).sum()),
                          "recall across borders": ((scores[n] >= default_threshold[n]) & ~np.isnan(scores[n]) & xb).sum() / max(len(xb_all), 1)}
                      for n in compared}).T.round(3))

In [ ]:
# Palette: validated categorical slots 1-4 (blue, orange, aqua, yellow) in a fixed order; aqua and yellow are
# low-contrast on white, so every line is also labelled directly and the table above gives the same numbers.
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
def short(name):
    return "Decision model" if name.startswith("Model ") else HYBRID if name.startswith(HYBRID) else name


shown = [NF_HEURISTIC, LEARNED] + model_names[:1] + [n for n in scores if n.startswith(HYBRID)][:1]
fig, ax = plt.subplots(figsize=(7.5, 4.6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ends = []
for name, color in zip(shown, SLOTS, strict=False):
    ok = ~np.isnan(scores[name])
    thr, precision, recall = P.pr_curve(y[ok], scores[name][ok])
    ax.plot(recall, precision, color=color, lw=2, label=short(name), zorder=2)
    i = int(np.argmin(np.abs(thr - default_threshold[name])))
    ax.plot(recall[i], precision[i], "o", ms=8, color=color, mec=SURFACE, mew=2, zorder=3)
    ends.append([name, color, float(precision[-1])])
ends.sort(key=lambda e: e[2])
for k in range(1, len(ends)):  # nudge end labels apart
    ends[k][2] = max(ends[k][2], ends[k - 1][2] + 0.06)
for name, _color, yy in ends:
    ax.annotate(short(name), (1.0, yy), xytext=(6, 0), textcoords="offset points", va="center",
                fontsize=9, color=INK2, annotation_clip=False)
ax.axhline(y.mean(), color=INK2, lw=1, ls=(0, (4, 3)), zorder=1)
ax.annotate("chance", (0.0, y.mean()), xytext=(4, 4), textcoords="offset points", fontsize=8, color=INK2)
ax.set(xlim=(0, 1), ylim=(0, 1.03), xlabel="Recall", ylabel="Precision")
ax.set_title("Precision-recall on candidate pairs (dots: default threshold)", loc="left", fontsize=11, color=INK)
ax.grid(color=GRID, lw=0.8, zorder=0)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color(GRID)
ax.tick_params(colors=INK2, labelsize=9)
ax.xaxis.label.set_color(INK2)
ax.yaxis.label.set_color(INK2)
ax.legend(loc="lower left", frameon=False, fontsize=9, labelcolor=INK2)
fig.subplots_adjust(right=0.78)
plt.show()


## 5. Where each scorer fails

Rows are slices of the candidate pairs; each scorer shows `recall / false merges` at its default threshold.
*Hazard codes differ* holds cascades (a cyclone and its floods); *coarse* sources give only a country and a day.

In [ ]:
slices = {
    "all candidates": np.ones(len(blocked), bool),
    "coarse - coarse": (blocked.pair_type == "coarse-coarse").to_numpy(),
    "coarse - precise": (blocked.pair_type == "coarse-precise").to_numpy(),
    "precise - precise": (blocked.pair_type == "precise-precise").to_numpy(),
    "hazard codes differ (cascade)": blocked.hazard_rel.isin(["cascade", "different"]).to_numpy(),
}
rows = {}
for slice_name, m in slices.items():
    row = {"positives": int((y[m] == 1).sum()), "negatives": int((y[m] == 0).sum())}
    for name, s in scores.items():
        pred = (s >= default_threshold[name]) & ~np.isnan(s)
        row[name] = f"{pred[m & (y == 1)].sum() / max((y[m] == 1).sum(), 1):.0%} / {int(pred[m & (y == 0)].sum())}"
    rows[slice_name] = row
display(pd.DataFrame(rows).T)


In [ ]:
titles = {i: r.title[:48] for i, r in records.items()}
view = blocked[["a", "b", "y"]].assign(title_a=blocked.a.map(titles), title_b=blocked.b.map(titles)).join(pd.DataFrame(scores))
decided = {n: (scores[n] >= default_threshold[n]) for n in scores}
wrong = pd.Series(decided[NF_HEURISTIC] != (y == 1))
cols = ["title_a", "title_b", "y"] + [n for n in scores if not n.startswith((HYBRID, "Rule:"))]

print(f"Heuristic false merges ({int((wrong & (y == 0)).sum())} in all), most confident first:")
display(view[wrong & (y == 0)].sort_values(NF_HEURISTIC, ascending=False)[cols].head(12).round(2))
print(f"Heuristic misses ({int((wrong & (y == 1)).sum())} in all), lowest score first:")
display(view[wrong & (y == 1)].sort_values(NF_HEURISTIC)[cols].head(12).round(2))


In [ ]:
# Does the model fix what the heuristic gets wrong, and break what it gets right?
for name in model_names:
    ok = ~np.isnan(scores[name])
    heuristic_right = (decided[NF_HEURISTIC] == (y == 1))[ok]
    model_right = (decided[name] == (y == 1))[ok]
    print(name)
    display(pd.crosstab(pd.Series(heuristic_right, name="heuristic"), pd.Series(model_right, name="model"))
            .rename(index={True: "right", False: "wrong"}, columns={True: "right", False: "wrong"}))

    # What a deployment would do with the model: link / drop / send to a person.
    verdict = [P.gate(None if np.isnan(p) else p) for p in scores[name]]
    gates = pd.crosstab(pd.Series(verdict, name=f"gate ({P.SAME_NO} / {P.SAME_YES})"),
                        pd.Series(np.where(y == 1, "same disaster", "different"), name="truth"))
    display(gates)

    relation_col = f"relation|{model_of[name]}"
    cascades = blocked[blocked.kind.eq("cascade") & (blocked.y == 1)]
    if relation_col in blocked and len(cascades):
        print("Relation the model gives to positive pairs of cascade disasters:")
        display(cascades[relation_col].value_counts(dropna=False).to_frame("pairs"))


## Confidence: does P(same) mean what it says?

To store a link's confidence in the catalog, a score of 0.8 should be right about 80% of the time. Per bin of
predicted P(same): how many pairs, their mean score and the share that really is one disaster. The Brier
score sums it up (0 is perfect, 0.25 a coin flip). Listwise pairs beyond the date gate count as 0.

Stability: `stability_pairs` pairs asked again, and asked with the two records in the other order.

In [ ]:
cal_rows, brier_rows = [], []
for n in [LEARNED, *model_names, *listwise_names]:
    for row in P.calibration_table(y, scores[n], bins=5, w=w):
        cal_rows.append({"scorer": n, **row})
    brier_rows.append({"scorer": n, "Brier score": P.brier(y, scores[n], w)})
calibration = pd.DataFrame(cal_rows)
display(pd.DataFrame(brier_rows).set_index("scorer").round(3))
display(calibration.set_index(["scorer", "P(same) bin"]).round(3))

stability = []
if stability_pairs and model_names:
    pick = np.random.default_rng(0).choice(len(blocked), min(stability_pairs, len(blocked)), replace=False)
    sample = [(records[blocked.a[i]], records[blocked.b[i]]) for i in pick]
    for name in model_names:
        first = scores[name][pick]
        for label_, kwargs in (("asked again", {}), ("records swapped", {"swap": True})):
            res = P.score_with_model(sample, model_of[name], api_key, **kwargs)
            again = np.array([np.nan if r["p_same"] is None else r["p_same"] for r in res])
            diff = np.abs(again - first)
            stability.append({"scorer": name, "check": label_, "pairs": len(pick),
                              "max |Δ P(same)|": float(np.nanmax(diff)), "mean |Δ P(same)|": float(np.nanmean(diff)),
                              "decisions changed at 0.5": int(((again >= 0.5) != (first >= 0.5)).sum()),
                              "cost": round(sum(r.get("cost") or 0 for r in res), 5)})
    display(pd.DataFrame(stability).set_index(["scorer", "check"]).round(3))

## From pairs to events

The user-facing result: every record of a disaster gathered under one event. Records are grouped by following
links (two records are linked when the scorer says they are the same disaster). *B-cubed precision*: of the
records grouped with a record, the share from its own disaster; *recall*: of its disaster's records, the share
grouped with it. One wrong link merges two whole events, so this is stricter than the pair scores.

*With source ids*: records that share a source's own disaster id are also linked (EM-DAT's DisNo and the GLIDE
number repeat on every affected country's record; IDMC's IDU and GIDD copies; GDACS episodes). These are the
sources' own links, not invented criteria. They join one disaster's records across countries, where the
records' text often names no storm and no method can tell from words alone.

In [ ]:
ids = sorted(records)
links = P.same_source_links(ids)
event_rows, assembled = [], {}
for n in ["Source ids only", *compared]:
    for with_ids in ((True,) if n == "Source ids only" else (False, True)):
        a_ = ([] if n == "Source ids only" else list(blocked.a)) + ([a for a, _ in links] if with_ids else [])
        b_ = ([] if n == "Source ids only" else list(blocked.b)) + ([b for _, b in links] if with_ids else [])
        s_ = ([] if n == "Source ids only" else list(scores[n])) + ([1.0] * len(links) if with_ids else [])
        ev = P.assemble_events(ids, a_, b_, s_, default_threshold.get(n, 0.5))
        assembled[(n, with_ids)] = ev
        bp, br, bf = P.bcubed(ev, label)
        event_rows.append({"scorer": n, "source ids": "with" if with_ids else "without",
                           "events formed": len(set(ev.values())), "gold disasters": len(set(label.values())),
                           "B-cubed precision": bp, "B-cubed recall": br, "B-cubed F1": bf})
events_table = pd.DataFrame(event_rows).set_index(["scorer", "source ids"])
print(f"{len(links)} links from shared source ids.")
display(events_table.round(3))

# Event cards: what a user would see for three disasters, with each record's strongest link inside its event.
card_scorer = (listwise_names or model_names or [LEARNED])[0]
ev = assembled[(card_scorer, True)]
strongest = {}
for a, b, s in zip(blocked.a, blocked.b, scores[card_scorer], strict=True):
    if not np.isnan(s) and s >= default_threshold[card_scorer] and ev[a] == ev[b]:
        strongest[a], strongest[b] = max(strongest.get(a, 0), s), max(strongest.get(b, 0), s)
names = {d["id"]: d["name"] for d in gold["disasters"]}
cards = []
for seed in ("emdat-event-2023-0054-TUR", "emdat-event-2023-0600-MAR", "emdat-event-2025-0917-JAM"):
    if seed not in ev:
        continue
    members = sorted((i for i in ev if ev[i] == ev[seed]), key=lambda i: (records[i].start, i))
    in_gold = sum(1 for d in label.values() if d == label[seed])
    card = pd.DataFrame({
        "source": [records[i].family for i in members],
        "starts": [records[i].start.date().isoformat() for i in members],
        "countries": [",".join(records[i].countries) for i in members],
        "title": [records[i].title[:60] for i in members],
        "link confidence": [round(strongest.get(i, np.nan), 2) for i in members],
        "gold": ["same disaster" if label.get(i) == label[seed] else f"other: {label.get(i)}" for i in members],
    })
    cards.append(card.assign(event=names[label[seed]]))
    print(f"{names[label[seed]]} ({card_scorer}, with source ids): {len(members)} records grouped; the gold disaster has {in_gold}.")
    display(card)

### Aftershocks (not scored)

Whether an aftershock belongs to its mainshock's disaster is an open definition, so the gold set leaves the
aftershocks of these destructive earthquakes out. Here each one is asked about against the records of its
mainshock's disaster, to show how each method would treat it once the definition is settled. GDACS cites a
GLIDE number for some: the Davao M6.7 shares its mainshock's number; the Kunar aftershocks have their own.

In [ ]:
after_ids = [a["id"] for a in gold["ambiguous"] if "aftershock" in a["why"] or "second shock" in a["why"]]
after_items = P.fetch_items(api_url, os.environ.get("STAC_API_KEY", ""), after_ids)
eq_disasters = [d for d in gold["disasters"] if any(m in records and records[m].glide == "EQ" for m in d["members"])]
diag = []
for aid in after_ids:
    if aid not in after_items:
        continue
    rec = P.record_from_item(after_items[aid])
    near = []  # earthquake disasters up to 30 days before it, nearest first
    for d in eq_disasters:
        ms = [records[m] for m in d["members"] if m in records]
        first = min(r.start for r in ms if r.start)
        km = min((P.haversine_km(rec.point, r.point) for r in ms if r.point and rec.point), default=np.inf)
        if first <= rec.start and (rec.start - first).days <= 30 and km <= 300:
            near.append((km, d, first))
    if not near:
        continue
    km, d, first = min(near, key=lambda t: t[0])
    others = [records[m] for m in d["members"] if m in records and records[m].family != rec.family]
    mainshock = min((records[m].start for m in d["members"] if m in records and not records[m].coarse), default=first)
    row = {"record": aid, "title": rec.title[:45], "hours after mainshock": round((rec.start - mainshock).total_seconds() / 3600, 1),
           "mainshock disaster": d["id"], "km to nearest record": round(km), "pairs": len(others),
           "heuristic: share linked": np.mean([P.heuristic_score(*P.pair_anchor(rec, o)) >= P.STRONG for o in others]),
           "CQL2 1 day: share linked": np.mean([P.cql2_match(rec, o, 1) for o in others])}
    for name in model_names:
        res = P.score_with_model([(rec, o) for o in others], model_of[name], api_key)
        ps = [r["p_same"] for r in res if r.get("p_same") is not None]
        row[f"{name}: mean P(same)"] = float(np.mean(ps)) if ps else np.nan
        row[f"{name}: relation"] = pd.Series([r.get("relation") for r in res]).mode().iloc[0] if res else None
    diag.append(row)
aftershocks = pd.DataFrame(diag)
display(aftershocks.round(2))

## Silver set: labels from the sources themselves

`silver_disasters.yml` is written by `build_silver.py` from the references GDACS publishes for its Orange and
Red alerts: the GLIDE number of the disaster and, for earthquakes, the USGS event. Montandon does not ingest these
fields, so no scorer can see them, and no person or model chose these labels. One disaster per GDACS event, with
its USGS event and every country record of its GLIDE number. The learned scorer is trained on the gold set
only. These pairs are mostly easy (an alert and the entries it cites), so they test scale and agreement with the
sources more than hard cases; negatives come from neighbouring alerts of the same hazard and country.

In [ ]:
silver_metrics, silver_path = pd.DataFrame(), Path("silver_disasters.yml")
sv_summary = {}
if silver and silver_path.exists():
    sv = yaml.safe_load(silver_path.read_text())
    sv_label, sv_windows, _ = P.gold_labels(sv)
    sv_items = P.fetch_items(api_url, os.environ.get("STAC_API_KEY", ""), list(sv_label))
    sv_label = {i: d for i, d in sv_label.items() if i in sv_items}
    sv_records = {i: P.record_from_item(x) for i, x in sv_items.items()}
    sv_pairs = pd.DataFrame(P.build_pairs(sv_records, sv, block_days))
    sv_pairs["cross_border"] = [bool(known(sv_records[a]) and known(sv_records[b]) and not known(sv_records[a]) & known(sv_records[b]))
                                for a, b in zip(sv_pairs.a, sv_pairs.b, strict=True)]
    sv_b = sv_pairs[sv_pairs.block == "ok"].reset_index(drop=True)
    sv_y, sv_w = sv_b.y.to_numpy(), P.pair_weights(sv_b.a, sv_b.b, P.copy_counts(sv_records, sv_label))
    sv_scores = {NF_HEURISTIC: sv_b.heuristic.astype(float).to_numpy(),
                 LEARNED: P.fit_logreg(X, y, l2).predict(sv_b[[f"f_{k}" for k in P.FEATURES]].to_numpy(float))}
    for rule_name, rule in P.RULES.items():
        sv_scores[rule_name] = np.array([rule(sv_records[a], sv_records[b]) for a, b in zip(sv_b.a, sv_b.b, strict=True)], dtype=float)
    sv_gap = sv_b.gap_days.to_numpy(float)
    for name in model_names:
        if len(sv_b) > max_model_calls:
            print(f"{len(sv_b)} silver pairs, above max_model_calls: skipping {name}.")
            continue
        res = P.score_with_model([(sv_records[a], sv_records[b]) for a, b in zip(sv_b.a, sv_b.b, strict=True)], model_of[name], api_key)
        sv_scores[name] = np.array([np.nan if r["p_same"] is None else r["p_same"] for r in res])
        model_log.append({"requested": model_of[name], "set": "silver", "pairs": len(res),
                          "errors": sum(1 for r in res if r.get("error")), "cost": round(sum(r.get("cost") or 0 for r in res), 5)})
        if listwise:
            gated = list(zip(sv_b.a[sv_gap <= gate_days], sv_b.b[sv_gap <= gate_days], strict=True))
            sv_groups = P.listwise_groups(gated, sv_records, max_candidates)
            res = P.score_listwise(sv_groups, sv_records, model_of[name], api_key)
            sc, _ = P.pair_scores_from_listwise(sv_groups, res, list(zip(sv_b.a, sv_b.b, strict=True)))
            sv_scores[f"Listwise {name.removeprefix('Model ')}"] = np.where(sv_gap > gate_days, 0.0, sc)
            listwise_log.append({"requested": model_of[name], "set": "silver", "requests": len(sv_groups),
                                 "errors": sum(1 for r in res if r.get("error")), "cost": round(sum(r.get("cost") or 0 for r in res), 5)})
    sv_names = [n for n in compared if n in sv_scores]
    samples = P.bootstrap_f1(sv_y, {n: (sv_scores[n], default_threshold[n]) for n in sv_names}, sv_b.window.to_numpy(), sv_w, n=bootstrap_samples)
    sv_best_rule = max(P.RULES, key=lambda n: P.prf(sv_y, sv_scores[n] >= 0.5, sv_w)[2])
    rows = []
    for n in sv_names:
        s, thr = sv_scores[n], default_threshold[n]
        ok = ~np.isnan(s)
        p_, r_, f_ = P.prf(sv_y[ok], s[ok] >= thr, sv_w[ok])
        lo, hi = P.interval(samples[n])
        dlo, dhi = P.interval(samples[n] - samples[sv_best_rule])
        xb_ = sv_b.cross_border.to_numpy() & (sv_y == 1)
        rows.append({"scorer": n, "pairs": int(ok.sum()), "precision": p_, "recall": r_, "F1": f_,
                     "F1 95% interval": f"{lo:.2f} to {hi:.2f}", "false merges": int(((s >= thr) & ok & (sv_y == 0)).sum()),
                     "Δ F1 vs best rule": f_ - P.prf(sv_y, sv_scores[sv_best_rule] >= 0.5, sv_w)[2],
                     "Δ F1 95% interval": f"{dlo:+.2f} to {dhi:+.2f}",
                     "recall across borders": ((s >= thr) & ok & xb_).sum() / max(int(xb_.sum()), 1)})
    silver_metrics = pd.DataFrame(rows).set_index("scorer")
    sv_summary = {"disasters": len(sv["disasters"]), "records": len(sv_records), "missing": len(set(P.gold_labels(sv)[0]) - set(sv_items)),
                  "candidates": len(sv_b), "positives": int(sv_y.sum()), "windows with negatives": int(sv_b[sv_b.y == 0].window.nunique()),
                  "positives lost to blocking": int(((sv_pairs.y == 1) & (sv_pairs.block != "ok")).sum()), "best rule": sv_best_rule}
    print(sv_summary)
    display(silver_metrics.round(3))
    display(sv_b.groupby(["source_a", "source_b"]).y.agg(pairs="size", positives="sum"))
elif silver:
    print("silver_disasters.yml not found: run build_silver.py to build it.")

## Cost at catalog scale

What running the chosen method on the catalog would cost, from the cost per call measured above. A sample of
the catalog (`scale_sample_days` days ending `scale_sample_end`) gives the in-scope candidate pairs a day: at
least one record from an impact source, the same hazard or a cascade, at most `gate_days` apart and near
enough. The cost follows the number of new records, not how often the job runs: hourly costs the same as daily.
The history estimate multiplies the impact-source records in the catalog by the in-scope pairs each produced
in the sample, so it is rough (older years have fewer sources, which makes it an upper bound).

In [ ]:
import json as _json
from datetime import UTC, datetime, timedelta

import requests

cost = {}
if scale_sample_days and model_names:
    end = datetime.fromisoformat(scale_sample_end).replace(tzinfo=UTC) + timedelta(days=1)
    start = end - timedelta(days=scale_sample_days)
    crowd = [P.record_from_item(x) for x in P.fetch_context(api_url, os.environ.get("STAC_API_KEY", ""), None,
                                                            start - timedelta(days=gate_days), end + timedelta(days=gate_days), max_items=200_000)]
    crowd = {r.id: r for r in crowd if r.start}
    new = [r for r in crowd.values() if start <= r.start < end and P.destructive(r)]
    by_family = {}
    for r in crowd.values():
        by_family.setdefault(r.family, []).append(r)
    in_scope = set()
    for a in new:
        for fam, rivals in by_family.items():
            if fam == a.family:
                continue
            for b in rivals:
                if P.gap_days(a, b, by_day=True) <= gate_days and P.hazard_relation(a, b) != "different" and P.block(a, b, gate_days) == "ok":
                    in_scope.add(tuple(sorted((a.id, b.id))))
    in_scope = sorted(in_scope)
    model = model_of[model_names[0]]

    def chars(requests):
        return sum(len(_json.dumps(r)) for r in requests)

    # Measured cost per character of request, from the gold runs above, applied to the sample's requests.
    gold_pair_chars = chars(P.decision_request(model, records[a], records[b]) for a, b in zip(blocked.a, blocked.b, strict=True))
    usd_per_char = {"pairwise": model_log[0]["cost"] / gold_pair_chars}
    sample_chars = {"pairwise": chars(P.decision_request(model, crowd[a], crowd[b]) for a, b in in_scope)}
    if listwise_names:
        usd_per_char["listwise"] = listwise_log[0]["cost"] / chars(P.listwise_request(model, records[a], [records[i] for i in c]) for a, c in groups)
        sample_groups = P.listwise_groups(in_scope, crowd, max_candidates)
        sample_chars["listwise"] = chars(P.listwise_request(model, crowd[a], [crowd[i] for i in c]) for a, c in sample_groups)
    try:
        stats = requests.get(api_url.rsplit("/stac", 1)[0] + "/stats/sources", timeout=60).json()
        impact_records = sum(s["events"] for s in stats if P.FAMILIES.get(s["source"], s["source"]) in P.IMPACT_SOURCES)
    except Exception:
        impact_records = None
    per_record = len(in_scope) / max(len(new), 1)
    for kind, c in sample_chars.items():
        per_day = c * usd_per_char[kind] / scale_sample_days
        cost[kind] = {"USD per day": round(per_day, 3), "USD per year": round(per_day * 365, 1),
                      "USD for the history (rough)": round(impact_records * per_record * c * usd_per_char[kind] / max(len(in_scope), 1), 0) if impact_records else None}
    cost["sample"] = {"days": scale_sample_days, "impact-source records per day": round(len(new) / scale_sample_days, 1),
                      "in-scope pairs per day": round(len(in_scope) / scale_sample_days), "catalog records in the sample": len(crowd),
                      "impact-source records in the catalog": impact_records}
    display(pd.DataFrame(cost).T)

## 6. Outputs

In [ ]:
out = Path(output_dir)
scored = pd.DataFrame({**scores, **variants}).join(blocked[["a", "b"] + [c for c in blocked if c.startswith(("relation|", "listwise_best|"))]])
(pairs.merge(scored, on=["a", "b"], how="left")
      .assign(title_a=lambda d: d.a.map(titles), title_b=lambda d: d.b.map(titles))
      .to_csv(out / "pair_scores.csv", index=False))
metrics.round(4).to_csv(out / "metrics.csv")
(out / "run_summary.json").write_text(json.dumps({
    "catalog": api_url, "gate_days": gate_days, "margin": margin, "distractors": distractors, "listwise": listwise_log,
    "records": len(records), "missing": missing, "pairs": len(pairs), "candidates": len(blocked), "positives": total_pos,
    "blocking_recall": 1 - len(lost) / max(total_pos, 1), "block_days": block_days, "models": model_log,
    "describe_detail": describe_detail,
}, indent=2))
for name, table in (("headline.csv", headline), ("calibration.csv", calibration), ("events.csv", events_table),
                    ("aftershocks.csv", aftershocks), ("silver_metrics.csv", silver_metrics)):
    table.to_csv(out / name)
if cards:
    pd.concat(cards).to_csv(out / "event_cards.csv", index=False)
(out / "cost.json").write_text(json.dumps({"cost": cost, "stability": stability, "silver": sv_summary}, indent=2, default=str))
print("wrote", ", ".join(sorted(p.name for p in out.iterdir())))


## What to do with this

* Judge methods on the comparison table (in scope, held-out first) and on *average precision*, which needs no
  threshold. A gain whose Δ F1 interval stays above 0 is more than noise on this gold set.
* Set a decision model's link threshold from the calibration table, not at 0.5: a scorer whose high bins are
  right more often than its scores say (underconfident) can link at a lower score. Re-check whenever the model
  version changes.
* Group events with the source ids switched on: EM-DAT and GLIDE already tie a disaster's country records
  together, and no text-based method can.
* Read the false merges and misses tables. Each row is either a scorer weakness or a wrong label: fix the
  label in `gold_disasters.yml` (with a review-log line), or fix the scorer.
* The aftershock table shows how each method would treat aftershocks; settle the definition before scoring them.
* The `pair_scores.csv` output has every pair with every score, for review in a spreadsheet.
